# 🏆 Neurogolf 2026: ARC Task Logic & Complexity Analysis

### 📋 Notebook Objectives:
1. **Analyze Complexity:** Identify the "low-hanging fruit" tasks that can be solved with minimal layers.
2. **Track Dimensionality:** Map out which tasks require expensive `Resize` or `TransposeConv` operators versus standard `Conv2D`.
3. **Isolate Primitives:** Find the most common logical operations to prioritize building reusable ONNX sub-graphs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set professional plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})

# Load the dataset (Update path if your dataset directory name differs)
# Using the corrected author URL format: karnakbayevartur
DATA_PATH = '/kaggle/input/datasets/karnakbaevarthur/neurogolf-2026-task-transformation-library/arc_primitives.csv'

try:
    df = pd.read_csv(DATA_PATH)
    print(f"✅ Dataset loaded successfully! Shape: {df.shape}")
except FileNotFoundError:
    print("⚠️ Dataset not found. Please ensure it is attached to the notebook.")
    # Fallback to local working directory if just generated
    df = pd.read_csv('/kaggle/working/arc_primitives.csv')

display(df.head())

---
## 1. Complexity & Operator Cost Distribution
The `Estimated_Complexity` score (1-10) is our proxy for network cost. 
* **Scores 1-3:** Likely solvable with 1x1 or 3x3 convolutions, pooling, or simple `Roll`/`Slice` operations.
* **Scores 7-10:** Require deep feature extraction, long-range dependencies, or heavy recurrent-like unrolled structures.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Complexity Distribution
sns.histplot(df['Estimated_Complexity'], bins=10, kde=True, ax=axes[0], color='indigo')
axes[0].set_title('Distribution of Task Complexity (1-10)', fontweight='bold')
axes[0].set_xlabel('Estimated Complexity (Proxy for ONNX Cost)')
axes[0].set_ylabel('Number of Tasks')

# Plot 2: Total Transformations per Task
sns.countplot(data=df, x='Total_Transformations', ax=axes[1], palette='viridis')
axes[1].set_title('Number of Primitives Required per Task', fontweight='bold')
axes[1].set_xlabel('Total Transformations Used')
axes[1].set_ylabel('Number of Tasks')

plt.tight_layout()
plt.show()

# Insight generation
easy_tasks = df[df['Estimated_Complexity'] <= 3].shape[0]
print(f"💡 STRATEGY INSIGHT: There are {easy_tasks} tasks with a complexity of 3 or lower. These should be targeted first for maximum leaderboard points at minimal parameter cost.")

---
## 2. The Dimensionality Bottleneck (Grid Resizing)
In ONNX, preserving spatial dimensions is computationally cheap (`Conv2D` with `padding='same'`). Changing dimensions requires dynamic cropping, upsampling, or transposed convolutions, which heavily impact MACs and memory.

Let's see how often the output grid size changes compared to the input, broken down by the primary logic category.

In [ ]:
plt.figure(figsize=(10, 6))

# Create a stacked bar chart of Grid_Size_Changed by Primary_Category
resize_counts = df.groupby(['Primary_Category', 'Grid_Size_Changed']).size().unstack(fill_value=0)
resize_counts.plot(kind='bar', stacked=True, color=['#4C72B0', '#C44E52'], figsize=(10, 6))

plt.title('Grid Resizing Requirements by Logic Category', fontweight='bold')
plt.xlabel('Primary Transformation Category')
plt.ylabel('Number of Tasks')
plt.legend(title='Grid Size Changed', labels=['False (Static Size)', 'True (Dynamic Size)'])
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

static_size_pct = (df['Grid_Size_Changed'] == False).mean() * 100
print(f"💡 STRATEGY INSIGHT: {static_size_pct:.1f}% of tasks keep the same grid size. You can reuse a standard static-shape CNN backbone for the majority of the dataset.")

---
## 3. Unpacking the Primitives: What exactly are we building?
To design modular ONNX blocks, we need to know which operations are requested most frequently. Let's parse the `All_Used_Transformations` column to rank the exact logical primitives found across the 400 tasks.

In [ ]:
from collections import Counter

# Split the pipe-separated strings and flatten the list
all_primitives = df['All_Used_Transformations'].dropna().str.split(' | ').explode()
# Clean up any potential whitespace
all_primitives = all_primitives.str.strip()
all_primitives = all_primitives[all_primitives != '']

# Count occurrences
primitive_counts = Counter(all_primitives)
top_primitives = pd.DataFrame(primitive_counts.most_common(15), columns=['Primitive', 'Frequency'])

plt.figure(figsize=(12, 6))
sns.barplot(data=top_primitives, x='Frequency', y='Primitive', palette='mako')
plt.title('Top 15 Most Frequently Used Logical Primitives', fontweight='bold')
plt.xlabel('Number of Tasks Using this Primitive')
plt.ylabel('Logical Primitive')

# Add values on the bars
for index, value in enumerate(top_primitives['Frequency']):
    plt.text(value + 2, index, str(value), va='center', fontsize=10)

plt.tight_layout()
plt.show()

---
## 4. Execution Plan: The "Low-Hanging Fruit" Pipeline
Based on our EDA, the optimal engineering pipeline prioritizes tasks that do **not** require grid resizing, use **fewer than 3** transformations, and have a **complexity of 3 or lower**. 

Below is the generated target list for your first ONNX architecture sprint. These tasks will yield the highest $max(1, 25 - \ln(cost))$ scores because their minimal logic translates to tiny models.

In [ ]:
# Filter for optimal first-targets
target_list = df[
    (df['Grid_Size_Changed'] == False) & 
    (df['Estimated_Complexity'] <= 3) & 
    (df['Total_Transformations'] <= 2)
].sort_values('Estimated_Complexity')

print(f"🎯 Found {len(target_list)} high-priority 'Easy' tasks.")
display(target_list[['Task_ID', 'Primary_Category', 'All_Used_Transformations', 'Estimated_Complexity']].head(15))

# Export this specific target list for your local ONNX testing pipeline
target_list.to_csv('onnx_sprint_1_targets.csv', index=False)
print("Saved top targets to 'onnx_sprint_1_targets.csv'")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set professional plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})

# Create a 1x2 figure
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Plot 1: Correlation Heatmap ---
cols_to_correlate = ['Spatial_Count', 'Object_Count', 'Color_Count', 'Pattern_Count', 'Estimated_Complexity']
corr_matrix = df[cols_to_correlate].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap = sns.diverging_palette(250, 20, as_cmap=True)

sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
            square=True, linewidths=.5, annot=True, fmt=".2f", cbar_kws={"shrink": .8}, ax=axes[0])
axes[0].set_title('Correlation: What Drives ONNX Complexity?', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

# --- Plot 2: Complexity Spread by Primary Category ---
sns.boxenplot(data=df, x='Primary_Category', y='Estimated_Complexity', palette='Set2', ax=axes[1])
sns.stripplot(data=df, x='Primary_Category', y='Estimated_Complexity', color='black', alpha=0.3, size=3, ax=axes[1])

axes[1].set_title('ONNX Implementation Cost by Primary Logic Type', fontweight='bold')
axes[1].set_xlabel('Primary Transformation Category')
axes[1].set_ylabel('Estimated Complexity (1-10)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

# --- Insights Section (Fixed) ---
print("\n" + "="*60)
print("🚀 ARCHITECTURE INSIGHTS GENERATED")
print("="*60)

# 1. Average Complexity Insight
if 'Primary_Category' in df.columns:
    avg_comp = df.groupby('Primary_Category')['Estimated_Complexity'].mean().sort_values()
    print(f"1. EASIEST ARCHITECTURE: '{avg_comp.index[0]}' tasks ({avg_comp.iloc[0]:.2f}/10).")
    print(f"2. HARDEST ARCHITECTURE: '{avg_comp.index[-1]}' tasks ({avg_comp.iloc[-1]:.2f}/10).")

# 2. Correlation Insight (Fixing the NameError)
# First, isolate the correlations with complexity
corrs = corr_matrix['Estimated_Complexity'].drop('Estimated_Complexity')

if not corrs.empty:
    max_corr_feature = corrs.idxmax()
    max_corr_val = corrs.max()
    print(f"3. PRIMARY COST DRIVER: '{max_corr_feature}' (r={max_corr_val:.2f}).")
    print(f"   -> Focus on optimizing {max_corr_feature.split('_')[0].lower()} ops to reduce cost.")
print("="*60)